# Crowd Management System - YOLO26m Training Pipeline

This notebook trains an Ultralytics **YOLO26 Medium (yolo26m)** model on custom crowd surveillance data with anti-overfitting regularizations.

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics with native YOLO26 support
!pip install --upgrade "ultralytics>=8.4.0"

In [ ]:
# Step 3: Mount Google Drive (optional, for persistent storage)
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Step 4: Clone repository or extract dataset
# !git clone https://github.com/vik05h/Crowd-Management-System.git
# %cd Crowd-Management-System

In [ ]:
# Step 5: Verify dataset configuration
import yaml
from pathlib import Path

data_config = {
    'path': '/content/Crowd-Management-System/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['person']
}

with open('colab_data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print('[INFO] Created colab_data.yaml successfully')

In [ ]:
# Step 6: Train YOLO26m with Anti-Overfitting Regularization
from ultralytics import YOLO

# Initialize pretrained YOLO26m
model = YOLO('yolo26m.pt')

# Run training
results = model.train(
    data='colab_data.yaml',
    epochs=100,
    patience=15,          # Early stopping to halt training if validation metric plateaus
    batch=16,             # Appropriate batch size for Colab T4/A100
    imgsz=1024,           # High-resolution input for detecting small crowd targets
    device=0,
    optimizer='auto',     # Uses YOLO26 MuSGD/AdamW
    cos_lr=True,          # Cosine annealing learning rate scheduler
    dropout=0.15,         # Dropout regularization to prevent parameter co-adaptation
    weight_decay=0.001,   # L2 weight decay to penalize large weights
    mosaic=0.5,           # Context augmentation
    mixup=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    close_mosaic=10,      # Disable mosaic in final 10 epochs for fine-tuning on natural frames
    project='runs/detect',
    name='yolo26m_custom',
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)

print('[SUCCESS] Training completed.')

In [ ]:
# Step 7: View training curve and metrics
from IPython.display import Image, display
results_img = Path('runs/detect/yolo26m_custom/results.png')
if results_img.exists():
    display(Image(str(results_img)))
else:
    print('[INFO] results.png not found yet.')

In [ ]:
# Step 8: Download trained weights
from google.colab import files
best_weight = Path('runs/detect/yolo26m_custom/weights/best.pt')
if best_weight.exists():
    files.download(str(best_weight))
    print('[INFO] Downloading best.pt to your computer.')
else:
    print('[WARN] best.pt not found.')